In [1]:
from typing import Union, Optional, Callable
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Lab 05: Hidden Markov Models

In this session, we will implement the algorithms presented in the lecture as solutions to the various tasks related to the Hidden Markov Models (HMMs): Filtering, Smoothing and Decoding.

## Part 1: Defining the HMM

We have prepared an implementation of a hidden Markov model. If you are interested, you can have a look at the file `hmm.py`, but this is not required to proceed with the lab.

In [2]:
from hmm import HiddenMarkovModel, make_grid_hmm

The implementation supports both continuous and discrete observation spaces. An HMM with a **discrete** observation space can be created as follows:
```python
HiddenMarkovModel(S, O, A, psi, None, pi)
```
where
* `S` is a list of states,
* `O` is a list of observations,
* `A` is the matrix of state transition probabilities $a_{ij} = p(z_j \mid z_i)$,
* `psi` is the matrix of observation emission probabilities $\psi_{ij} = p(x_j \mid z_i)$,
* `pi` is a vector describing the initial state distribution.

When the observation space is **continuous**, we do not encode `psi` as a matrix (as it would have to have uncountably many rows), but as a function `psi_fun`, which maps an observation $x \in \mathbb R^n$ to an $\lvert \mathcal S \rvert$-dimensional vector $\psi(x)$ with $\psi(x)_i = p(x \mid z_i)$.
The HMM is then created as follows:
```python
HiddenMarkovModel(S, None, A, None, psi_fun, pi)
```

In this session, we will consider two simple HMMs.

The first HMM describes the behaviour of a person (named Bob) depending on the weather. Bob can choose his activity depending on the weather (sunny or rainy): either going for a walk, shopping or cleaning inside. 

The weather transitions are as follows: 
* When it is raining on some day, it is raining the following day with probability 0.7
* When it is sunny on some day, it is sunny the following day with probability 0.6

Bob decides his daily activity as follows:
* When it is raining on some day, Bob goes for a walk with probability 0.1, goes shopping with probability 0.4 and stays inside cleaning with probability 0.5. 
* When it is sunny on some day, Bob goes for a walk with probability 0.6, goes shopping with probability 0.3 and stays inside cleaning with probability 0.1. 


### Question 1

Implement the corresponding HMM.

### <font color='green'><u>Solution</u></font>

In [3]:
states = ['rainy', 'sunny']
observations = ['walk', 'shop', 'clean']

A = np.array([
    [0.7, 0.3],
    [0.4, 0.6]
])

psi = np.array([
    [0.1, 0.4, 0.5],
    [0.6, 0.3, 0.1]
])

pi = np.array([0.5, 0.5])
weather_hmm = HiddenMarkovModel(states, observations, A, psi, None, pi)

The second HMM corresponds to some artificial situation where we observe the (noisy) sum of two integers. These two integers are chosen through a random walk on a bounded subset of $\mathbb Z^2$.

The HMM is specified as follows:
* The state space $\mathcal{S}$ is a set of tuples corresponding to a finite grid: $\mathcal{S} = \lbrace - N, \dotsc, N \rbrace^2$ for some integer $N > 0$
* The state transition corresponds to a random walk on the grid: from a state $(i,j)$, one can transition with equiprobability to any state $(i^\prime, j^\prime)$, if $\lvert i' - i\rvert + \lvert j' - j\rvert = 1$
* The initial distribution is uniform over the grid
* The observation space $\mathcal{O}$ is the space of real numbers: $\mathcal{O} = \mathbb{R}$
* The emission probabilities are normal distributions: $p(x \mid z = (i,j)) = \mathcal{N}(x; i + j, \sigma^2)$

This HMM is already implemented for you.

In [4]:
N = 2
sigma = 0.1
grid_hmm = make_grid_hmm(N, sigma)

## Part 2: Filtering

The goal of filtering is to compute the probabilities $\alpha_t(j) = p(z_t = j \mid x_1, \dotsc, x_t)$. This is done throughout the forwards algorithm.

### Question 2 (*)

a) Implement the forwards algorithm.<br>

### <font color='green'><u>Solution</u></font>

In [10]:
def normalize(p: np.ndarray):
    Z = p.sum()
    return p / Z, Z

def forward(hmm: HiddenMarkovModel, x: np.ndarray):
    T = len(x)
    psi = hmm.get_psi(x)
    n = len(hmm.S)

    alphas = np.empty((T, n))
    Z = np.empty(T)
    alphas[0], Z[0] = normalize(psi[:, 0] * hmm.pi)
    for t in range(1, T):
        alphas[t], Z[t] = normalize(psi[:, t] * (hmm.A.T @ alphas[t-1]))
    return alphas, np.log(Z)

b) We consider the following observations for the two tasks. Apply the forward algorithm to these two observations. Do the outcomes look consistent? Compute also the probability $p(x_1, \dotsc, x_T)$.

### <font color='green'><u>Solution</u></font>

In [6]:
# 1. Observations for weather HMM
x_weather = [0, 0, 2, 1, 0]
forward(weather_hmm, x_weather)

(array([[0.14285714, 0.85714286],
        [0.11698113, 0.88301887],
        [0.79385844, 0.20614156],
        [0.70162716, 0.29837284],
        [0.2071165 , 0.7928835 ]]),
 array([-1.04982212, -0.97135051, -1.29448946, -1.01110771, -1.22160763]))

The observations for the grid HMM are generated from a spiral which starts at the origin $(0,0)$ and moves outward, visiting every state. From this, the sums are computed and random noise is applied.

In [8]:
# 2. Observations for grid HMM
spiral = [(0,0), (0,1), (-1,1), (-1,0), (-1,-1), (0,-1), (1,-1), (1,0), (1,1), (1,2), (0,2), (-1,2), (-2,2), (-2,1), (-2,0), (-2,-1), (-2,-2), (-1,-2), (0,-2), (1,-2), (2,-2), (2,-1), (2,0), (2,1), (2,2)]
x_grid = [norm.rvs(loc = i+j, scale = sigma) for (i,j) in spiral]

forward(grid_hmm, x_grid)

(array([[0.00000000e+000, 2.74602026e-208, 6.64141595e-096,
         5.97543372e-027, 2.00000000e-001, 2.74602026e-208,
         6.64141595e-096, 5.97543372e-027, 2.00000000e-001,
         2.49024667e-019, 6.64141595e-096, 5.97543372e-027,
         2.00000000e-001, 2.49024667e-019, 1.15347065e-080,
         5.97543372e-027, 2.00000000e-001, 2.49024667e-019,
         1.15347065e-080, 1.98757049e-185, 2.00000000e-001,
         2.49024667e-019, 1.15347065e-080, 1.98757049e-185,
         0.00000000e+000],
        [0.00000000e+000, 0.00000000e+000, 1.59966021e-232,
         6.51194102e-095, 1.26814764e-044, 0.00000000e+000,
         1.37113733e-232, 4.34129402e-095, 2.21925836e-044,
         3.00000000e-001, 1.59966021e-232, 4.34129402e-095,
         1.90222145e-044, 2.00000000e-001, 1.41488867e-037,
         6.51194102e-095, 2.21925836e-044, 2.00000000e-001,
         1.21276172e-037, 1.18732982e-160, 1.26814764e-044,
         3.00000000e-001, 1.41488867e-037, 1.18732982e-160,
         0.00

## Part 3: Smoothing

The goal of smoothing is to compute the probabilities $\gamma_t(j) = p(z_t = j \mid x_1, \dotsc, x_T)$. Smoothing is solved using the forwards-backwards algorithm.

### Question 3 (*)

a) Implement the forwards-backwards algorithm.<br>
b) Test the algorithm on the $x$'s we introduced in the previous part.

### <font color='green'><u>Solution</u></font>

In [11]:
def backward(hmm: HiddenMarkovModel, x: np.ndarray):
    T = len(x)
    psi = hmm.get_psi(x)
    n = len(hmm.S)

    betas = np.empty((T, n))
    Z = np.empty(T)
    betas[T-1] = np.ones(n)
    Z[T-1] = 1
    for t in reversed(range(T-1)):
        betas[t], Z[t] = normalize(hmm.A @ (psi[:, t+1] * betas[t+1]))
    return betas, np.log(Z)

def forward_backward(hmm: HiddenMarkovModel, x: np.ndarray):
    alphas, _ = forward(hmm, x)
    betas, _ = backward(hmm, x)
    gammas = alphas * betas
    return gammas / gammas.sum(axis=1, keepdims=True)

In [12]:
forward_backward(weather_hmm, x_weather)

array([[0.10090221, 0.89909779],
       [0.16105985, 0.83894015],
       [0.78470223, 0.21529777],
       [0.59509165, 0.40490835],
       [0.2071165 , 0.7928835 ]])

In [13]:
forward_backward(grid_hmm, x_grid)

array([[0.00000000e+000, 0.00000000e+000, 1.89374038e-189,
        0.00000000e+000, 2.11826544e-001, 0.00000000e+000,
        1.31374687e-189, 0.00000000e+000, 1.97043364e-001,
        0.00000000e+000, 1.89374038e-189, 0.00000000e+000,
        1.82260184e-001, 0.00000000e+000, 1.51522492e-080,
        0.00000000e+000, 1.97043364e-001, 0.00000000e+000,
        1.05115887e-080, 0.00000000e+000, 2.11826544e-001,
        0.00000000e+000, 1.51522492e-080, 0.00000000e+000,
        0.00000000e+000],
       [0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
        6.89700981e-095, 0.00000000e+000, 0.00000000e+000,
        0.00000000e+000, 3.95622523e-095, 0.00000000e+000,
        3.17739816e-001, 0.00000000e+000, 3.95622523e-095,
        0.00000000e+000, 1.82260184e-001, 0.00000000e+000,
        6.89700981e-095, 0.00000000e+000, 1.82260184e-001,
        0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
        3.17739816e-001, 0.00000000e+000, 0.00000000e+000,
        0.00000000e+000],
    

## Part 4: Decoding

Decoding corresponds to the task of finding the optimal sequence of states explaining an observation. 

A first method for decoding consists in maximizing the posterior marginals. This corresponds to selecting, for each time step $t$, the state $j$ maximizing $p(z_t = j \mid x_1, \dotsc, x_T)$. 

### Question 4

Implement the posterior marginals maximization (MPM). Compute the optimal sequence of states for the given two sequences of observations.

### <font color='green'><u>Solution</u></font>

In [14]:
def mpp_decoding(hmm: HiddenMarkovModel, x: np.ndarray):
    gammas = forward_backward(hmm, x)
    z_mpp = gammas.argmax(axis = 1)
    return z_mpp

In [15]:
weather_mpp = mpp_decoding(weather_hmm, x_weather)
[weather_hmm.S[i] for i in weather_mpp]

['sunny', 'sunny', 'rainy', 'rainy', 'sunny']

In [16]:
grid_mpp = mpp_decoding(grid_hmm, x_grid)
[grid_hmm.S[i] for i in grid_mpp]

[(-2, 2),
 (-1, 2),
 (-1, 1),
 (-2, 1),
 (-2, 0),
 (-1, 0),
 (0, 0),
 (0, 1),
 (1, 1),
 (1, 2),
 (1, 1),
 (0, 1),
 (0, 0),
 (-1, 0),
 (-1, -1),
 (-2, -1),
 (-2, -2),
 (-2, -1),
 (-1, -1),
 (-1, 0),
 (0, 0),
 (0, 1),
 (1, 1),
 (1, 2),
 (2, 2)]

The problem with the MPM decoding is that it does not maximize the probability of the whole sequence, and might therefore lead to sub-optimal choices. For this reason, why prefer using the Viterbi algorithm, which relies on dynamic programming to compute the maximum a posteriori estimator of the latent states $z_1, \dotsc, z_T$.

### Question 5 (*)

Implement the Viterbi algorithm. Compare the results with the MPM decoding.

### <font color='green'><u>Solution</u></font>

In [17]:
def viterbi(hmm: HiddenMarkovModel, x: np.ndarray):
    T = len(x)
    psi = hmm.get_psi(x)
    n = len(hmm.S)

    # Initialization
    delta = np.empty((T,n))
    a = np.empty((T,n))
    delta[0, :] = hmm.pi * psi[:, 0]
    a[0, :] = 0

    # Recursion
    for t in range(1, T):
        temp = delta[t-1].reshape(-1, 1) * psi[:, t] * hmm.A
        delta[t] = np.max(temp, axis=0)
        a[t] = np.argmax(temp, axis=0)

    # Termination
    z = np.empty(T)
    z[-1] = np.argmax(delta[-1])
    for t in reversed(range(T-1)):
        z[t] = a[t+1, int(z[t+1])]
    return z

### Question 6

Look at the solution returned by the Viterbi algorithm for the grid HMM. How well does it recover the spiral that generated the observations? Can you explain the behavior?

### <font color='green'><u>Solution</u></font>

In [18]:
weather_viterbi = viterbi(weather_hmm, x_weather)
[weather_hmm.S[int(i)] for i in weather_viterbi]

['sunny', 'sunny', 'rainy', 'rainy', 'sunny']

In [19]:
grid_viterbi = viterbi(grid_hmm, x_grid)
[grid_hmm.S[int(i)] for i in grid_viterbi]

[(-2, 2),
 (-1, 2),
 (-2, 2),
 (-2, 1),
 (-2, 0),
 (-2, 1),
 (-2, 2),
 (-1, 2),
 (0, 2),
 (1, 2),
 (0, 2),
 (-1, 2),
 (-2, 2),
 (-2, 1),
 (-2, 0),
 (-2, -1),
 (-2, -2),
 (-2, -1),
 (-2, 0),
 (-2, 1),
 (-2, 2),
 (-1, 2),
 (0, 2),
 (1, 2),
 (2, 2)]

The spiral is not recovered well at all. This is not surprising, since all integer pairs $(i,j)$ with the same sum are considered equivalent. The equivalence classes are diagonals in $\mathbb Z^2$. Therefore, at each step, there are usually several directions that would generate the same observations.